In [0]:
%sql
CREATE OR REPLACE TABLE electronics_retailer_clg.gold.fact_sales AS
SELECT
    s.order_number,
    s.line_item,
    s.order_date,
    d.year,
    d.month,
    s.customerkey,
    s.productkey,
    s.storekey,
    s.quantity,
    p.unit_price_usd,
    er.exchange,
    ROUND(
        (s.quantity * p.unit_price_usd) / er.exchange,
        2
    ) AS revenue_usd,
    DATEDIFF(s.delivery_date, ifnull(s.order_date,s.delivery_date)) AS delivery_days,
    CASE 
        WHEN s.storekey = 0 THEN 'online'
        ELSE 'store'
    END AS channel,
    s.currency_code

FROM electronics_retailer_clg.silver.sales as s
LEFT JOIN electronics_retailer_clg.gold.dim_products as p
    ON s.productkey = p.productkey
LEFT JOIN electronics_retailer_clg.gold.dim_sales d
    ON s.order_date = d.date
LEFT JOIN electronics_retailer_clg.gold.dim_exchange_rate er
    ON s.currency_code = er.currency
    AND s.order_date = er.date;
     

In [0]:
# %sql
# CREATE OR REPLACE TABLE electronics_retailer_clg.gold.fact_sales AS
# SELECT 
#   s.order_number,
#   s.order_date,
#   s.delivery_date,
#   CASE 
#     WHEN s.delivery_date IS NULL THEN NULL
#     ELSE DATEDIFF(s.delivery_date, s.order_date)
#   END as delivery_time_days,
#   s.customerkey,
#   c.gender,
#   c.continent,
#   s.storekey,
#   st.country as store_country,
#   st.channel,
#   s.productkey,
#   p.category as product_category,
#   p.unit_price_usd,
#   s.quantity,
#   s.currency_code,
#   er.exchange_rate as exchange_rate,
#   -- Revenue calculation in USD
#   (s.quantity * p.unit_price_usd * COALESCE(er.exchange_rate, 1.0)) as revenue_usd
# FROM electronics_retailer_clg.silver.sales s
# LEFT JOIN electronics_retailer_clg.gold.dim_customers c ON s.customerkey = c.customer_key
# LEFT JOIN electronics_retailer_clg.gold.dim_products p ON s.productkey = p.productkey
# LEFT JOIN electronics_retailer_clg.gold.dim_stores st ON s.storekey = st.store_key
# LEFT JOIN electronics_retailer_clg.silver.exchange_rates er 
#   ON s.currency_code = er.currency 
#   AND s.order_date = er.date
# WHERE s.order_date IS NOT NULL;

In [0]:
%sql
select * from electronics_retailer_clg.gold.fact_sales

In [0]:

sales = spark.read.table("electronics_retailer_clg.silver.sales")
products = spark.read.table("electronics_retailer_clg.silver.products")
exchange = spark.read.table("electronics_retailer_clg.silver.exchange_rates")

In [0]:

prod_join = sales.alias("s").join(products.alias("p"), on="productkey",how="left").select("s.productkey","s.quantity","s.order_date","p.product_name","p.unit_cost_usd","p.unit_price_usd","s.currency_code")
display(prod_join)

In [0]:

from pyspark.sql import functions as F
from pyspark.sql import functions as F
exchange = prod_join.alias("p").join(
    exchange.alias("e"), 
    (F.col("p.currency_code") == F.col("e.currency")) & (F.col("p.order_date") == F.col("e.date")),
    how="left"
)

display(exchange)

In [0]:

display(
    exchange.groupBy(
        F.year("order_date").alias("year"),
        F.month("order_date").alias("month")
    )
    .agg(
        F.sum((F.col("unit_price_usd") * F.col("quantity")) /  F.col("exchange"))
         .cast("decimal(18,2)")
         .alias("total_price")
    )
    .filter(F.col("year") == 2020)
    .orderBy("year","month")
)